# Data Validation

Data validation is the process of checking whether the data follows the expected rules, formats, types, ranges, and business constraints.

Validation helps us identify incorrect or unrealistic values before using the data for analysis or Machine Learning.

In this notebook, we will validate the Titanic dataset using different types of validation rules such as range, type, format, category, null, business rule, referential, date, and constraint validation.

### Real-World Example

For a customer dataset:
- Age should not be negative.
- Salary should not be negative.
- Gender should belong to an accepted set.
- Percentage should be between 0 and 100.

Similarly, in the Titanic dataset, values such as `Age`, `Fare`, `Sex`, `Embarked`, and `Survived` can be checked against reasonable rules.

### AI/ML Relevance

Data validation helps prevent incorrect values from entering the Machine Learning pipeline and improves the reliability of the data used for model training.

In [1]:
import pandas as pd

df = pd.read_csv("Titanic-Dataset.csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset Shape: (891, 12)

Columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Range Validation

Range validation checks whether numerical values fall within an expected range.

### Problem

Numerical columns may contain unrealistic values.

### Analysis

For example, `Age` cannot logically be negative, and `Fare` should not be negative.

### Technique Selected

We use conditional checks to identify values outside the expected range.

### Reason

Range validation is simple and appropriate when the valid minimum and maximum values are known.

### Implementation

For the Titanic dataset:
- Age should be greater than or equal to 0.
- Fare should be greater than or equal to 0.
- Pclass should be between 1 and 3.
- Survived should be either 0 or 1.

### Result

The validation output shows whether invalid values are present.

### Impact

Invalid numerical values can affect statistics, visualizations, and Machine Learning model performance.

In [2]:
print("Invalid Age values:", (df["Age"] < 0).sum())
print("Invalid Fare values:", (df["Fare"] < 0).sum())
print("Invalid Pclass values:", (~df["Pclass"].isin([1, 2, 3])).sum())
print("Invalid Survived values:", (~df["Survived"].isin([0, 1])).sum())

Invalid Age values: 0
Invalid Fare values: 0
Invalid Pclass values: 0
Invalid Survived values: 0


## Type Validation

Type validation checks whether each column has the expected data type.

### Problem

A numerical column may sometimes be stored as text, or a categorical column may contain unexpected data types.

### Analysis

Machine Learning algorithms and preprocessing techniques depend on correct data types.

### Technique Selected

We inspect the data types using Pandas.

### Reason

Checking `dtypes` provides a quick way to identify columns that may require type correction.

### Implementation

We inspect the data type of every column in the Titanic dataset.

### Result

The output shows the current data type of each column.

### Impact

Incorrect data types can cause errors during calculations, encoding, scaling, and model training.

In [3]:
print("Column Data Types:")
print(df.dtypes)

Column Data Types:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


## Format Validation

Format validation checks whether values follow an expected structure or pattern.

### Problem

Text fields may contain empty or incorrectly formatted values.

### Analysis

For example, a name field should contain meaningful text rather than an empty value.

### Technique Selected

We check whether important text fields contain blank or empty values.

### Reason

This is useful for identifying values that are technically present but not meaningful.

### Implementation

We check the `Name` and `Ticket` columns for blank strings.

### Result

The output shows how many blank values are found.

### Impact

Incorrectly formatted values may create problems during feature engineering and categorical processing.

In [4]:
blank_names = df["Name"].astype(str).str.strip().eq("").sum()
blank_tickets = df["Ticket"].astype(str).str.strip().eq("").sum()

print("Blank Name values:", blank_names)
print("Blank Ticket values:", blank_tickets)

Blank Name values: 0
Blank Ticket values: 0


## Category Validation

Category validation checks whether categorical values belong to an accepted set of categories.

### Problem

Categorical columns can contain unexpected values because of data-entry errors.

### Analysis

In the Titanic dataset, `Sex` should contain valid categories such as `male` and `female`. The `Embarked` column should contain valid port codes such as `S`, `C`, and `Q`.

### Technique Selected

We compare the existing values with the expected categories.

### Reason

This allows unexpected categories to be detected before encoding.

### Implementation

We validate the `Sex` and `Embarked` columns.

### Result

The output identifies values that are outside the accepted categories.

### Impact

Unexpected categories can create inconsistent features and may cause problems during Machine Learning preprocessing.

In [5]:
valid_sex = {"male", "female"}
valid_embarked = {"S", "C", "Q"}

invalid_sex = df.loc[
    df["Sex"].notna() & ~df["Sex"].isin(valid_sex), "Sex"
]

invalid_embarked = df.loc[
    df["Embarked"].notna() & ~df["Embarked"].isin(valid_embarked), "Embarked"
]

print("Invalid Sex values:")
print(invalid_sex.unique())

print("\nInvalid Embarked values:")
print(invalid_embarked.unique())

Invalid Sex values:
<ArrowStringArray>
[]
Length: 0, dtype: str

Invalid Embarked values:
<ArrowStringArray>
[]
Length: 0, dtype: str


## Null Validation

Null validation checks whether required values are missing.

### Problem

Missing values can occur because information was not collected, was unavailable, or was lost during data processing.

### Analysis

The Titanic dataset contains missing values in some columns, such as `Age`, `Cabin`, and `Embarked`.

### Technique Selected

We use `isnull()` and calculate the number and percentage of missing values.

### Reason

Before deciding how to handle missing values, we should first identify where they occur and how significant they are.

### Implementation

We calculate missing-value counts and percentages for every column.

### Result

The output shows the columns containing missing values.

### Impact

Missing values can affect statistical analysis and Machine Learning models. The appropriate treatment should be decided during the missing-value preprocessing stage.

In [6]:
null_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2)
})

print(null_summary)

             Missing Count  Missing Percentage
PassengerId              0                0.00
Survived                 0                0.00
Pclass                   0                0.00
Name                     0                0.00
Sex                      0                0.00
Age                    177               19.87
SibSp                    0                0.00
Parch                    0                0.00
Ticket                   0                0.00
Fare                     0                0.00
Cabin                  687               77.10
Embarked                 2                0.22


## Business Rule Validation

Business rule validation checks whether the data follows rules based on the meaning of the dataset.

### Problem

A value can have the correct data type but still be logically incorrect.

### Analysis

For example, an age value of `-5` is numeric, but it is not logically valid.

For the Titanic dataset:
- Age should not be negative.
- Fare should not be negative.
- Passenger class should be 1, 2, or 3.
- Survived should be 0 or 1.
- Family-related counts such as `SibSp` and `Parch` should not be negative.

### Technique Selected

We create explicit business rules and count the records that violate them.

### Reason

Business rules are based on the meaning of the data rather than only its technical data type.

### Result

The output shows the number of records violating each rule.

### Impact

Business-rule validation helps prevent logically incorrect data from affecting analysis and Machine Learning.

In [7]:
business_rule_results = {
    "Negative Age": (df["Age"] < 0).sum(),
    "Negative Fare": (df["Fare"] < 0).sum(),
    "Invalid Pclass": (~df["Pclass"].isin([1, 2, 3])).sum(),
    "Invalid Survived": (~df["Survived"].isin([0, 1])).sum(),
    "Negative SibSp": (df["SibSp"] < 0).sum(),
    "Negative Parch": (df["Parch"] < 0).sum()
}

for rule, count in business_rule_results.items():
    print(f"{rule}: {count}")

Negative Age: 0
Negative Fare: 0
Invalid Pclass: 0
Invalid Survived: 0
Negative SibSp: 0
Negative Parch: 0


## Referential Validation

Referential validation checks whether values correctly refer to records or identifiers in another related dataset.

### Problem

In relational data, a record may contain an identifier that should exist in another table.

### Example

A customer order may contain a `Customer_ID`. That ID should exist in the customer table.

### Titanic Dataset

The Titanic dataset is a single standalone dataset and does not contain a separate parent table or foreign-key relationship that needs to be checked.

Therefore, a full referential-integrity check is not applicable to this dataset.

### Technique Selected

We inspect identifier columns and confirm that the dataset does not require a cross-table reference check.

### Result

No external reference table is required for the current Titanic dataset.

### Impact

Referential validation becomes important when multiple related datasets are combined because invalid references can lead to incorrect joins and Machine Learning features.

In [8]:
identifier_columns = [
    column for column in ["PassengerId", "Ticket"]
    if column in df.columns
]

print("Identifier-like columns in the Titanic dataset:")
print(identifier_columns)

print("\nNo external reference table is available for referential validation.")

Identifier-like columns in the Titanic dataset:
['PassengerId', 'Ticket']

No external reference table is available for referential validation.


## Date Validation

Date validation checks whether date values are correctly formatted and logically valid.

### Problem

Dates may contain invalid formats or logically impossible relationships.

### Example

In an employee dataset:
- Date of joining should not be before date of birth.
- A transaction date should follow the expected date format.

### Titanic Dataset

The Titanic dataset does not contain a dedicated date-of-birth, joining-date, or transaction-date column.

Therefore, direct date validation is not applicable to the current dataset.

### Technique Selected

We check whether the dataset contains date-like columns before applying date validation.

### Reason

Date validation should only be performed when date information is actually available.

### Result

The dataset can be checked for columns that have been stored using a datetime data type.

### Impact

Correct date validation is important in datasets where time-based information is used for analysis or Machine Learning.

In [9]:
datetime_columns = df.select_dtypes(include=["datetime64[ns]"]).columns.tolist()

print("Datetime columns:")
print(datetime_columns)

if not datetime_columns:
    print("\nNo datetime columns are present in the Titanic dataset.")

Datetime columns:
[]

No datetime columns are present in the Titanic dataset.


## Constraint Validation

Constraint validation checks whether data follows predefined constraints.

These constraints may be based on allowed values, ranges, uniqueness, or relationships between columns.

### Problem

A dataset may contain values that violate rules even though the values have the correct data type.

### Analysis

For the Titanic dataset, we can apply constraints such as:
- `PassengerId` should be unique.
- `Survived` should contain only 0 or 1.
- `Pclass` should contain only 1, 2, or 3.
- `Age`, `Fare`, `SibSp`, and `Parch` should not be negative.

### Technique Selected

We create multiple validation checks and summarize their results.

### Reason

Using several constraints together gives a broader view of the dataset's data quality.

### Result

The output shows whether each constraint is satisfied.

### Impact

Constraint validation helps maintain consistent and reliable data before Machine Learning preprocessing.

In [10]:
constraints = {
    "PassengerId is unique": df["PassengerId"].is_unique,
    "Survived contains only 0 and 1": df["Survived"].dropna().isin([0, 1]).all(),
    "Pclass contains only 1, 2 and 3": df["Pclass"].dropna().isin([1, 2, 3]).all(),
    "Age is non-negative": (df["Age"].dropna() >= 0).all(),
    "Fare is non-negative": (df["Fare"].dropna() >= 0).all(),
    "SibSp is non-negative": (df["SibSp"].dropna() >= 0).all(),
    "Parch is non-negative": (df["Parch"].dropna() >= 0).all()
}

for constraint, result in constraints.items():
    print(f"{constraint}: {result}")

PassengerId is unique: True
Survived contains only 0 and 1: True
Pclass contains only 1, 2 and 3: True
Age is non-negative: True
Fare is non-negative: True
SibSp is non-negative: True
Parch is non-negative: True


## Complete Validation Summary

After performing the validation checks, we can summarize the important findings.

### Problem

The purpose of validation is to identify data-quality issues before preprocessing rather than immediately changing the data.

### Analysis

The Titanic dataset was checked for numerical ranges, data types, text formats, accepted categories, null values, business rules, referential requirements, date information, and general constraints.

### Technique Selected

Python and Pandas validation rules were used to identify values that do not follow the expected conditions.

### Reason

These techniques are simple, transparent, and easy to reproduce during the preprocessing pipeline.

### Result

The validation process identifies potential data-quality issues without unnecessarily modifying the original dataset.

### Impact

The identified issues can be addressed in later preprocessing steps such as missing-value treatment, type conversion, encoding, and feature preparation.

### Insight

Data validation shows that checking whether data is technically valid is different from simply checking whether the dataset can be loaded. A value must also make sense according to the meaning and rules of the data.

### AI/ML Relevance

Reliable input data is important for Machine Learning because invalid values, incorrect categories, missing information, and violated constraints can negatively affect feature engineering, model training, and prediction quality.

### Important Note

This notebook focuses on **identifying and documenting validation problems**. Complete data correction or preprocessing should be performed in the appropriate later preprocessing stages rather than blindly changing the data here.

In [11]:
print("DATA VALIDATION SUMMARY")
print("=" * 30)

print("Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nExact Duplicate Records:", df.duplicated().sum())

print("\nValidation Checks:")
print("Invalid Age:", (df["Age"] < 0).sum())
print("Invalid Fare:", (df["Fare"] < 0).sum())
print("Invalid Pclass:", (~df["Pclass"].isin([1, 2, 3])).sum())
print("Invalid Survived:", (~df["Survived"].isin([0, 1])).sum())
print("Invalid Sex:", (~df["Sex"].dropna().isin(["male", "female"])).sum())
print("Invalid Embarked:", (~df["Embarked"].dropna().isin(["S", "C", "Q"])).sum())

print("\nValidation completed without modifying the original dataset.")

DATA VALIDATION SUMMARY
Dataset Shape: (891, 12)

Missing Values:
Age         177
Cabin       687
Embarked      2
dtype: int64

Exact Duplicate Records: 0

Validation Checks:
Invalid Age: 0
Invalid Fare: 0
Invalid Pclass: 0
Invalid Survived: 0
Invalid Sex: 0
Invalid Embarked: 0

Validation completed without modifying the original dataset.
